In [1]:
import pandas as pd
import numpy as np
import sys
sys.path.insert(0, "../../utils/")
from sklearn.model_selection import train_test_split
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import SMOTE
from training_models.classification_models import ClassificationModels
from joblib import dump
import optuna
import json
import os

c:\Users\hantr\anaconda3\envs\ML_Class\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def optimize_rf(trial, X_train, X_val, y_train, y_val):
    params = {
        "n_estimators": trial.suggest_int(f"n_estimators", 100, 1000),
        "criterion": trial.suggest_categorical(f"criterion", ["gini", "entropy", "log_loss"]),
        "min_samples_split": trial.suggest_int(f"min_samples_split", 2, 50),
        "min_samples_leaf": trial.suggest_int(f"min_samples_leaf", 1, 30),
        "max_features": trial.suggest_categorical(f"max_features", ["sqrt", "log2"]),
        "max_depth": trial.suggest_int(f"max_depth", 5, 50)
    }
    clf_model = ClassificationModels(X_train=X_train, X_val=X_val, y_train=y_train, y_val=y_val)
    clf_model.instance_random_forest(**params)
    clf_model.process_model(kfold=True, k=5)
    
    return clf_model.performances["validation_metrics"]["F1-score"]

In [3]:
def undersampling(df_data, seed):
    X = df_data.drop('target', axis=1)
    y = df_data['target']  
    #Se definen los objetos para submuestrear
    undersampler = RandomUnderSampler(sampling_strategy='not minority', random_state=seed)

    #Se aplica el submuestreo
    X_res, y_res= undersampler.fit_resample(X, y)
    df_resampled = pd.concat([X_res,y_res], axis=1)
    
    return df_resampled

In [4]:
def oversampling(df_data, seed):
    X = df_data.drop('target', axis=1)
    y = df_data['target']  
    #Se definen los objetos para sobremuestrear
    smote = SMOTE(random_state=seed)

    #Se aplica el sobremuestreo
    X_res, y_res= smote.fit_resample(X, y)
    df_resampled = pd.concat([X_res,y_res], axis=1)
    
    return df_resampled

In [5]:
def split(df_data, seed):
    #Separa los datos
    data_under= undersampling(df_data, seed)
    data_over= oversampling(df_data, seed)
    train_data, val_data = train_test_split(df_data, test_size=0.2, random_state=seed)
    train_data_under, val_data_under = train_test_split(data_under, test_size=0.2, random_state=seed)
    train_data_over, val_data_over = train_test_split(data_over, test_size=0.2, random_state=seed)
    return train_data, val_data, train_data_under, val_data_under, train_data_over, val_data_over

In [6]:
def train(train_v, validation_v, iteration, repr_name, div, seed):
    #Separa datos de sus target de entrenamiento y validacion
    train_values = train_v.drop(columns="target").values
    train_response = train_v["target"].values

    validation_values = validation_v.drop(columns="target").values
    validation_response = validation_v["target"].values

    print(f"Training model Random Forest, iteration: {iteration}")
    #Se instancia el objeto
    clf_model = ClassificationModels(X_train=train_values, X_val=validation_values, y_train=train_response, y_val=validation_response)
    #Se entrena el respectivo algoritmo con k-fold
    clf_model.instance_random_forest()
    clf_model.process_model(kfold=True, k=5)

    #Se guarda el modelo
    dump(clf_model.model, f"../../models/RandomForest_{div}_{iteration}_{repr_name}_seed{seed}.joblib")

    return clf_model.performances

In [7]:
def train_with_optuna(train_v, validation_v, iteration, repr_name, div, seed, n_trials=30):
    X_train = train_v.drop(columns="target").values
    y_train = train_v["target"].values
    X_val = validation_v.drop(columns="target").values
    y_val = validation_v["target"].values
    
    print(f"Optimizing Random Forest, iteration: {iteration}")
    
    study = optuna.create_study(direction="maximize")
    func = lambda trial: optimize_rf(trial, X_train, X_val, y_train, y_val)
    study.optimize(func, n_trials=n_trials)
    
    best_params = study.best_params
    print(f"Best params iteration {iteration}: {best_params}")
    
    #Entrenar modelo final con los mejores parámetros
    clf_model=ClassificationModels(X_train=X_train, X_val=X_val, y_train=y_train, y_val=y_val)
    clf_model.instance_random_forest(**best_params)
    clf_model.process_model(kfold=True, k=5)
    
    dump(clf_model.model, f"../../models/RandomForest_{div}_{iteration}_{repr_name}_seed{seed}_optuna.joblib")
    
    params_dir = f"../../models/best_params/"
    os.makedirs(params_dir, exist_ok=True)
    params_path = os.path.join(params_dir, f"best_params_{div}_{iteration}_{repr_name}_seed{seed}.json")
    with open(params_path, "w") as f:
        json.dump(best_params, f, indent=4)

    return clf_model.performances, best_params

In [8]:
rename_map = {
    "f1_weighted": "F1-score",
    "recall_weighted": "Recall",
    "precision_weighted": "Precision",
    "accuracy": "Accuracy"
}

In [9]:
def store_optuna_params(optuna_params_list, best_params_list, sampling_types, iteration, seed):  # NUEVO
    for params, sampling in zip(best_params_list, sampling_types):
        row = {
            "iteration": iteration,
            "seed": seed,
            "sampling": sampling
        }
        row.update(params)
        optuna_params_list.append(row)

In [10]:
def metrics(perf, iteration, seed, sampling):
    #Se obtienen las metricas de entrenamiento y validacion en variables diferentes
    train_metrics = perf["training_metrics"]
    val_metrics = perf["validation_metrics"]
    #Se elimina la matriz de confusiones
    val_metrics= val_metrics.copy()
    val_metrics.pop("Confusion Matrix", None)
    #Renombra metricas
    train_renamed = {rename_map.get(k, k): v for k, v in train_metrics.items()}
    #Se asignan los valores de las metricas a un diccionario
    row = {
        "iteration": iteration,
        "seed": seed,
        "sampling": sampling
    }
    for metric_name in rename_map.values():
        row[f"Train_{metric_name}"] = round(train_renamed[metric_name], 4)
        row[f"Val_{metric_name}"] = round(val_metrics[metric_name], 4)
    
    return row

In [11]:
def main_train(df_data, repr_name, unique_seeds, use_optuna=False):
    all_metrics = []
    optuna_params = []
    for i, seed in enumerate(unique_seeds):
        df_train, df_val, df_train_under, df_val_under, df_train_over, df_val_over = split(df_data, seed)
        
        if use_optuna:
            perf_base, best_params_base = train_with_optuna(df_train, df_val, i, repr_name, 'base', seed)
            perf_under, best_params_under = train_with_optuna(df_train_under, df_val_under, i, repr_name, 'undersampling', seed)
            perf_over, best_params_over = train_with_optuna(df_train_over, df_val_over, i, repr_name, 'oversampling', seed)
            store_optuna_params(optuna_params_list=optuna_params, best_params_list=[best_params_base, best_params_under, best_params_over], sampling_types=['base', 'undersampling', 'oversampling'], iteration=i, seed=seed)
        else:
            perf_base = train(df_train, df_val, i, repr_name, 'base', seed)
            perf_under = train(df_train_under, df_val_under, i, repr_name, 'undersampling', seed)
            perf_over = train(df_train_over, df_val_over, i, repr_name, 'oversampling', seed)
        
        all_metrics.append(metrics(perf_base, i, seed, 'base'))
        all_metrics.append(metrics(perf_under, i, seed, 'undersampling'))
        all_metrics.append(metrics(perf_over, i, seed, 'oversampling'))

    df_metrics = pd.DataFrame(all_metrics)
    df_metrics.to_csv(f"../../models/metrics_{repr_name}_RandomForest.csv", index=False)
    if use_optuna:
        df_optuna_params = pd.DataFrame(optuna_params)
        df_optuna_params.to_csv(f"../../models/optuna_params_{repr_name}_RandomForest.csv", index=False)

In [12]:
repr_name="ProtT5"
df_data = pd.read_csv(f"../../data/numerical_rep/{repr_name}.csv")
df_data.drop(["experimental_characteristics"], axis=1, inplace=True)

In [13]:
folder = "../../data/numerical_rep/"
unique_seeds= [42]
#unique_seeds = np.random.choice(range(100), size=30, replace=False)
#unique_seeds = [94, 42, 98, 43, 90, 44, 99, 93, 66, 34, 72, 60, 6, 39, 26, 74, 17,8, 51, 96, 53, 13, 20, 33, 29, 65, 46, 82, 79, 89]

In [14]:
print(f"Processing {repr_name}")
metrics_path = f"../../models/metrics_{repr_name}.csv"
seeds_used = unique_seeds
main_train(df_data, repr_name, seeds_used, use_optuna=True)
print(f"Finished processing {repr_name}")
print("=====================================")

Processing ProtT5


[I 2025-06-01 00:05:58,983] A new study created in memory with name: no-name-0d2ca16f-b353-41b2-9637-a77a90f98d42


Optimizing Random Forest, iteration: 0


[I 2025-06-01 00:07:04,781] Trial 0 finished with value: 0.5753114394121904 and parameters: {'n_estimators': 996, 'criterion': 'gini', 'min_samples_split': 16, 'min_samples_leaf': 26, 'max_features': 'sqrt', 'max_depth': 20}. Best is trial 0 with value: 0.5753114394121904.
[I 2025-06-01 00:08:24,160] Trial 1 finished with value: 0.5741908266546591 and parameters: {'n_estimators': 846, 'criterion': 'gini', 'min_samples_split': 34, 'min_samples_leaf': 5, 'max_features': 'sqrt', 'max_depth': 8}. Best is trial 0 with value: 0.5753114394121904.
[I 2025-06-01 00:09:14,796] Trial 2 finished with value: 0.4474597506068756 and parameters: {'n_estimators': 955, 'criterion': 'log_loss', 'min_samples_split': 23, 'min_samples_leaf': 5, 'max_features': 'log2', 'max_depth': 49}. Best is trial 0 with value: 0.5753114394121904.
[I 2025-06-01 00:09:22,530] Trial 3 finished with value: 0.4409898999414833 and parameters: {'n_estimators': 193, 'criterion': 'log_loss', 'min_samples_split': 27, 'min_samples_

Best params iteration 0: {'n_estimators': 1000, 'criterion': 'gini', 'min_samples_split': 30, 'min_samples_leaf': 22, 'max_features': 'sqrt', 'max_depth': 31}


...

Best params iteration 0: {'n_estimators': 553, 'criterion': 'entropy', 'min_samples_split': 43, 'min_samples_leaf': 19, 'max_features': 'sqrt', 'max_depth': 38}


[I 2025-06-01 00:54:06,350] A new study created in memory with name: no-name-c3c3c32d-b215-4e1d-85e1-1dc7d6504633


Optimizing Random Forest, iteration: 0


[I 2025-06-01 00:54:35,953] Trial 0 finished with value: 0.604660594220624 and parameters: {'n_estimators': 783, 'criterion': 'gini', 'min_samples_split': 35, 'min_samples_leaf': 29, 'max_features': 'log2', 'max_depth': 7}. Best is trial 0 with value: 0.604660594220624.
[I 2025-06-01 00:55:32,642] Trial 1 finished with value: 0.7007141178103979 and parameters: {'n_estimators': 388, 'criterion': 'gini', 'min_samples_split': 10, 'min_samples_leaf': 8, 'max_features': 'sqrt', 'max_depth': 37}. Best is trial 1 with value: 0.7007141178103979.
[I 2025-06-01 00:56:03,256] Trial 2 finished with value: 0.6126605573864259 and parameters: {'n_estimators': 568, 'criterion': 'gini', 'min_samples_split': 43, 'min_samples_leaf': 4, 'max_features': 'log2', 'max_depth': 43}. Best is trial 1 with value: 0.7007141178103979.
[I 2025-06-01 00:58:48,876] Trial 3 finished with value: 0.6647048086491573 and parameters: {'n_estimators': 944, 'criterion': 'log_loss', 'min_samples_split': 26, 'min_samples_leaf':

Best params iteration 0: {'n_estimators': 325, 'criterion': 'gini', 'min_samples_split': 15, 'min_samples_leaf': 30, 'max_features': 'sqrt', 'max_depth': 25}
Finished processing ProtT5
